# Computing Tempered Energy 

We aim to implement a parallel tempering scheme for annealed MCMC as per Du et al.'s [2024 paper](https://arxiv.org/pdf/2302.11552) on compositional generation with energy-based diffusion. In order to implement parallel tempering into the sampling regime of annealed MCMC, we aim to evaluate the difference between true tempered energy and our approximations.

In [56]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [57]:
training_samples_size = 1000
training_samples = torch.normal( 0.5,  0.25, (training_samples_size, 1))

Training process follows standard energy based model training as per [Yu et. al 2020](https://arxiv.org/pdf/1903.08689)

In [58]:
class MLP(nn.Module):
    def __init__(self, x_dim, t_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim + t_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=-1)
        return self.net(xt)

In [61]:
n_epochs = 10
batch_size = 32
alpha = 0.5
K = 5
step_size = 0.1

model = MLP(x_dim=1, t_dim=1, hidden_dim=64, output_dim=1)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

sigma = 1.0
buffer = []

def langevin(x, t):
    x = x.detach().requires_grad_(True)
    for _ in range(K):
        e = model(x, t).sum()
        g = torch.autograd.grad(e, x)[0]
        x = x - step_size * g + (2 * step_size) ** 0.5 * sigma * torch.randn_like(x)
        x = x.detach().requires_grad_(True)
    return x.detach()

for epoch in range(n_epochs):
    idx = torch.randint(0, training_samples.size(0), (batch_size,))
    x_pos = training_samples[idx]

    if len(buffer) >= batch_size and torch.rand(1) < 0.95:
        x0 = buffer[torch.randint(0, len(buffer), (batch_size,))]
    else:
        x0 = torch.randn_like(x_pos)

    t = torch.zeros(batch_size, 1)

    x_neg = langevin(x0, t)
    buffer.append(x_neg)

    E_pos = model(x_pos, t)
    E_neg = model(x_neg, t)

    loss = (alpha * (E_pos**2 + E_neg**2) + E_pos - E_neg).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Loss at epoch {epoch}: {loss.item()}")

Loss at epoch 0: 0.02649623714387417
Loss at epoch 1: 0.011596613563597202
Loss at epoch 2: -0.017856450751423836
Loss at epoch 3: -0.016245253384113312
Loss at epoch 4: -0.04309743270277977
Loss at epoch 5: -0.09104375541210175
Loss at epoch 6: -0.06078951060771942
Loss at epoch 7: -0.1326691210269928
Loss at epoch 8: -0.135393887758255
Loss at epoch 9: -0.1281888484954834
